#  Boundary Processing (Plymouth)

This notebook loads ONS boundary data (LAD, LSOA), filters to Plymouth,
clips to the local authority boundary, and saves the processed layers to a
single GeoPackage.

## 1. Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cartopy.crs as ccrs
import cartopy
import pyogrio
import geopandas as gpd


from pathlib import Path

PROJECT_DIR = next(candidate
                   for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                   if (candidate / "01_Data").exists()
                   )


## 2. Configure Paths

In [ ]:
#configure paths to the necessary data directories

RAW_BOUNDARY_DIR = PROJECT_DIR / "01_Data/Raw/ONS_Boundaries"
PROCESSED_BOUNDARY_DIR = PROJECT_DIR / "01_Data/Processed/Boundaries"

PROCESSED_BOUNDARY_DIR.mkdir(parents=True, exist_ok=True)

print("Raw boundary data:      ", RAW_BOUNDARY_DIR.resolve())
print("Processed boundary data:", PROCESSED_BOUNDARY_DIR.resolve())
print("Raw folder exists:      ", RAW_BOUNDARY_DIR.exists())
print("Processed folder exists:", PROCESSED_BOUNDARY_DIR.exists())


## 3. Load and Inspect Raw Data

In [ ]:
# Inspect layer names inside each GeoPackage

boundary_files = sorted(RAW_BOUNDARY_DIR.glob("*.gpkg"))

for file in boundary_files:
    layers = pyogrio.list_layers(file)
    print(file.name)
    print(layers)
    print()

In [ ]:
# Read in the LAD and LSOA boundaries
lad_2024 = gpd.read_file(RAW_BOUNDARY_DIR / "LAD_DEC_2024_UK_BGC.gpkg")
lsoa_2021 = gpd.read_file(RAW_BOUNDARY_DIR / "LSOA_DEC_2021_EW_BGC_V5.gpkg")

print("lad 2024: ", lad_2024.shape)
print("lsoa 2021:", lsoa_2021.shape)

In [ ]:
#check the coordinate reference systems (CRS) of the two GeoDataFrames
print("lad 2024: ", lad_2024.crs)
print("lsoa 2021:", lsoa_2021.crs)

## 4. Exploratory Checks

In [ ]:
# Check for invalid geometries
print("lad 2024: ", (~lad_2024.is_valid).sum())
print("lsoa 2021:", (~lsoa_2021.is_valid).sum())

In [ ]:
# Check geometry types
print("lad 2024: ", lad_2024.geom_type.value_counts().to_dict())
print("lsoa 2021:", lsoa_2021.geom_type.value_counts().to_dict())

## 5. Filter Plymouth Boundaries



In [ ]:
#filter for plymouth in lad 2024 (using the LAD24CD code for Plymouth)
code = "E06000026"
lad_plymouth = lad_2024[lad_2024["LAD24CD"] == code].copy()

print(lad_plymouth[["LAD24CD", "LAD24NM"]])


In [ ]:
#check the total area for plymouth
lad_plymouth_area_km2 = lad_plymouth.geometry.area.iloc[0]/1e6
print(lad_plymouth_area_km2)

In [ ]:
# Filter LSOAs within Plymouth boundary using representative points
plymouth_boundary = lad_plymouth.geometry.iloc[0]
lsoa_inside_plymouth = lsoa_2021.geometry.representative_point().within(plymouth_boundary)
plymouth_lsoa = lsoa_2021[lsoa_inside_plymouth].copy()


print(f"Plymouth LSOAs: {len(plymouth_lsoa)}")

## 6. Clip LSOA to Plymouth boundary

In [ ]:
# Clip LSOA to Plymouth boundary
plymouth_clipped_lsoa = gpd.clip(plymouth_lsoa, lad_plymouth)

print(plymouth_clipped_lsoa.shape)

In [ ]:
#plot the Plymouth LAD and LSOA boundaries
fig = plt.figure(figsize=(10, 10))
ax = plt.axes(projection=ccrs.epsg(27700))
plymouth_clipped_lsoa.plot(ax=ax, facecolor="none", edgecolor="blue", linewidth=0.5)
lad_plymouth.boundary.plot(ax=ax, edgecolor="black", linewidth=1)

ax.set_title("Plymouth LAD with 2021 LSOA Boundaries")
ax.set_axis_off()

## 7. Save Outputs

In [ ]:
#save the processed boundaries to a GeoPackage
processed_boundaries_gpkg = PROCESSED_BOUNDARY_DIR / "plymouth_boundaries.gpkg"

lad_plymouth.to_file(processed_boundaries_gpkg, layer="lad_plymouth_2024", driver="GPKG")
plymouth_clipped_lsoa.to_file(processed_boundaries_gpkg, layer="lsoa_plymouth_2021_clipped", driver="GPKG")
plymouth_lsoa.to_file(processed_boundaries_gpkg, layer="lsoa_plymouth_2021", driver="GPKG")

In [ ]:
#check saved layers
pyogrio.list_layers(processed_boundaries_gpkg)